# Run the VLM as the paper does

[04_01](../04/04_01_run_vlm.ipynb) ran LLaVA-OneVision-Qwen2-0.5B with the same fraction of rows
in every projection and with the rows in checkpoint order.
Here the model runs with what [05_01](05_01_calibrate.ipynb) prepared, as in the paper:
every projection reads its own number of rows, and the chunk selectors choose among rows in hot–cold order.

The questions, the selectors and the two recordings are those of 04_01:
whether the answers to video questions are right, and how long the selected rows take to read from flash.

The paper is [VLM in a flash](https://arxiv.org/abs/2511.18692); upstream is its code in `../../vlm-flash`.
Other inputs are the read profile from [01](../01/01_01_measure_chunk_latency.ipynb) and
the windows tuned for the paper's selector in [03_01](../03/03_01_tune_paper.ipynb).
Run from this directory on the target SSD.

In [9]:
from contextlib import ExitStack
from copy       import deepcopy
from math       import ceil
from pathlib    import Path
from tempfile   import TemporaryDirectory
from time       import perf_counter_ns
from zipfile    import ZipFile

from huggingface_hub import hf_hub_download
from transformers    import AutoProcessor, DynamicCache, LlavaOnevisionForConditionalGeneration

import os
import sys
import torch

import numpy  as np
import pandas as pd

from torch.utils.cpp_extension import load

In [10]:
UPSTREAM = Path("../../vlm-flash")

sys.path.insert(0, str((UPSTREAM / "src").resolve()))
sys.path.insert(0, str((UPSTREAM / "scripts").resolve()))

import vlmflash
from vlmflash._native import native
from _lock            import exclusive
from _vlm             import encode_video, load_frames

## 1. Model

The model is a vision tower followed by a language model.
It takes a video and a text, and writes a text.
Only the language model's weights are selected.

In [11]:
MODEL_ID       = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
MODEL_REVISION = "74dd0bf867a4cda7950c17663794267c60cf4b40"

In [12]:
# Inference only: no gradients below.
torch.set_grad_enabled(False)

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision = MODEL_REVISION,
)

net = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision = MODEL_REVISION,
    dtype    = torch.float16,
).to(device).eval()

Loading weights: 100%|██████████| 716/716 [00:00<00:00, 1973.28it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      


In [13]:
# The language model inside the vision-language wrapper.
decoder = net.model.language_model

The language model is a stack of blocks.
The attention and the MLP of a block are built from linear projections.
Every block has seven: `q_proj`, `k_proj`, `v_proj`, and `o_proj` in attention;
`gate_proj`, `up_proj`, and `down_proj` in the MLP.

Each input channel of a projection owns one row of the weight matrix stored on flash,
so choosing channels chooses the rows to read.

In [14]:
# projection name -> layer
projections = {}

for name, layer in decoder.named_modules():
    if name.endswith("_proj"):
        projections[name] = layer

len(projections)

168

## 2. Questions

TempCompass is one of the paper's benchmarks.
Each multiple-choice question asks about one short video.
It lists its options one per line, and its answer starts with the letter of the right option.

In [15]:
DATA_ID       = "lmms-lab/TempCompass"
DATA_REVISION = "c8a67d88a3bc6fd4b2f9ae2f9e112668fbe05722"

In [16]:
question_file = hf_hub_download(
    DATA_ID,
    "multi-choice/test-00000-of-00001.parquet",
    repo_type = "dataset",
    revision  = DATA_REVISION,
)

video_file = hf_hub_download(
    DATA_ID,
    "tempcompass_videos.zip",
    repo_type = "dataset",
    revision  = DATA_REVISION,
)

In [17]:
questions = pd.read_parquet(question_file)
archive   = ZipFile(video_file)

questions.head(3)

,video_id,question,answer,dim
0,1034419625,What is the man doing in the video?\nA. dunkin...,A. dunking a basketball,action
1,1034419625,Which activity is the man engaged in?\nA. pass...,C. dunking a basketball,action
2,1034419625,What action is the man performing?\nA. dribbli...,B. dunking a basketball,action


Use every question of the 100 videos of experiment 04. 05_01 calibrated on other videos.

In [18]:
VIDEOS = 100  # Videos sampled from the 410 of the benchmark

video_ids = (
    questions.video_id
    .drop_duplicates()
    .sample(VIDEOS, random_state=0)
)

questions = questions[questions.video_id.isin(video_ids)]
len(questions)

391

## 3. Video

Sample 32 frames uniformly over a video.

In [19]:
FRAMES = 32  # Frames sampled from each video

def load_video(video_id):
    """Return the sampled frames of one video as images."""

    # The frame loader reads from a file, so unpack the video first.
    with TemporaryDirectory() as folder:
        path = archive.extract(f"videos/{video_id}.mp4", folder)

        # load_frames also returns a flag that is True only for upstream's built-in clip.
        frames, _ = load_frames(path, FRAMES)

    return frames

The vision tower turns every frame into the same number of tokens,
each as wide as the language model's hidden state.

In [20]:
def encode(frames):
    """Return the frame tokens of one video: (frames, 1, tokens per frame, hidden width)."""

    video_tokens = encode_video(net, processor, frames, device)

    return video_tokens.reshape(len(frames), 1, -1, video_tokens.shape[-1])

The first video is the example of this notebook, as in 04_01, which shows its frames.

In [21]:
example_id     = video_ids.iloc[0]
example_tokens = encode(load_video(example_id))

example_tokens.shape

torch.Size([32, 1, 196, 896])

## 4. Prompt

The chat template of the model wraps a text and puts a placeholder where the video goes.

In [22]:
def write_prompt(text):
    """Return the chat prompt of one text, with a placeholder for the video."""

    content = [
        {"type" : "video"},
        {"type" : "text", "text" : text},
    ]

    conversation = [
        {"role" : "user", "content" : content},
    ]

    return processor.apply_chat_template(
        conversation,
        add_generation_prompt = True,
    )

In [23]:
print(write_prompt("Describe what happens in this video."))

<|im_start|>user <video>
Describe what happens in this video.<|im_end|><|im_start|>assistant



The model reads the part before the placeholder, then the video, then the part after it.
It expects one newline token between the video and the part after it.

In [24]:
embedding = decoder.get_input_embeddings()

def embed_text(text):
    """Return the token embeddings of a text: (1, tokens, hidden width)."""

    ids = processor.tokenizer(text, return_tensors="pt").input_ids

    return embedding(ids.to(device))

In [25]:
# Part before the placeholder, which is processor.video_token.
# It is the same for every text.
before, _ = write_prompt("").split(processor.video_token)
prefix    = embed_text(before)

# Newline token after the video: (1, 1, hidden width)
newline = net.model.image_newline[None, None]

In [26]:
def after_video(text):
    """Return what follows the video: the newline token, then the rest of the prompt."""

    _, after = write_prompt(text).split(processor.video_token)

    return torch.cat([newline, embed_text(after)], dim=1)

## 5. Calibration

A condition requests a fraction $f$ of the projection weights.
For each $f$, 05_01 searched every block for the number of rows $R$ that each of its projections reads,
as TEAL does. `budgets.csv` holds them.

In [27]:
budgets = pd.read_csv("budgets.csv")

# requested fraction -> (projection name -> R)
budget_of = {}

for fraction, rows in budgets.groupby("fraction"):
    budget_of[fraction] = rows.set_index("projection").R.to_dict()

05_01 also found the hot–cold order of the rows of every projection:
the rows that are important most often come first. `orders.npz` holds one order per projection.

Rows lie on flash in one of two layouts, the checkpoint order or the hot–cold order.

In [28]:
hot_cold = np.load("orders.npz")

# layout -> (projection name -> order). Row j on flash belongs to input channel order[j].
orders = {
    "checkpoint" : {},
    "hot-cold"   : {},
}

for name, layer in projections.items():
    orders["checkpoint"][name] = torch.arange(layer.in_features)
    orders["hot-cold"][name]   = torch.from_numpy(hot_cold[name])

## 6. Selectors

In a forward pass, a projection with $N$ input channels receives $x \in \mathbb{R}^{T \times N}$,
the activations of $T$ tokens. The importance of channel $i$, and of its weight row, is

$$
v_i = \frac{1}{T}
\sum_{t=1}^{T}|x_{t,i}|.
$$

In [29]:
def importance(x):
    """Return the importance of every input channel, on the CPU."""

    n = x.shape[-1]

    return (
        x
        .reshape(-1, n)

        .abs()
        .mean(dim=0)

        .float()
        .cpu()
    )

A selector receives the importance of the $N$ rows of one projection, in their order on flash,
and a budget of $R$ rows. It returns a mask over the rows. The methods are:

- `Dense` selects every row.
- `Top-R` selects the $R$ most important rows. It is the paper's baseline.
- `Paper greedy` is the paper's chunk selection. It may select fewer than $R$ rows.
- `Tiles` cuts the rows into tiles of $B$ consecutive rows, selects the most important tiles,
  and fills the rest of the budget with one more run.

As in the paper, the baseline reads rows in checkpoint order and the chunk selectors in hot–cold order.

In [30]:
# method -> layout of the rows it reads
LAYOUTS = {
    "Dense"        : "checkpoint",
    "Top-R"        : "checkpoint",
    "Paper greedy" : "hot-cold",
    "Tiles"        : "hot-cold",
}

Paper and Tiles estimate the cost of a read from the profile of
[01](../01/01_01_measure_chunk_latency.ipynb): the median latency of each profiled read size.
Let $z_*$ (`tile_reference_kib`) be the size with the lowest cost per KiB
and $w$ the size of one row in KiB.
Tiles use $B=\lceil z_*/w\rceil$. Paper uses the windows tuned in
[03_01](../03/03_01_tune_paper.ipynb). Both run as in [03_02](../03/03_02_measure_latency.ipynb).

In [31]:
PROFILE = Path("../01/chunk_latency_measurements.csv")
TUNING  = Path("../03/paper_tuning.csv")

In [32]:
measurements = pd.read_csv(PROFILE)

latencies = (
    measurements.groupby("size_kib")["latency_ms"]
    .median()
    .sort_index()
)

assert np.isfinite(latencies).all() and (latencies > 0).all()

tile_reference_kib = float((latencies / latencies.index).idxmin())
max_profiled_kib   = float(latencies.index.max())
latency_table      = vlmflash.LatencyTable(latencies.to_dict())

In [33]:
tuning = pd.read_csv(TUNING)
tuning = tuning.set_index(["n", "d"], verify_integrity=True)

Paper's selector is in upstream's native extension, which also holds the flash reader
that section 10 uses. Tiles is in [selectors.cpp](../03/selectors.cpp).

In [34]:
os.environ.setdefault("MAX_JOBS", "1")
torch.set_num_threads(1)

assert torch.cuda.is_available(), "Paper's measured GPU sort requires CUDA."

In [35]:
reader = native()
assert reader is not None, "Build the upstream native extension first."

In [36]:
selectors = load(
    "vlmflash_selection03_notebook",
    [str(Path("../03/selectors.cpp").resolve())],
    extra_cflags = ["-O3", "-std=c++17"],
    verbose      = True,
)

ninja: no work to do.


In [37]:
def paper_windows(n, w, tuned):

# 1. SINGLE-ROW FALLBACK
    if max_profiled_kib < 2 * w:
        return torch.tensor([1]), 1

# 2. ROW GEOMETRY
    start  = max(1, int(tuned.start_kb / w))
    stride = max(1, int(tuned.jump_cap_kb / w))
    stop   = min(n + 1, int((max_profiled_kib + 1) / w))

# 3. CANDIDATE WINDOWS
    rows = list(range(start, stop, start))
    assert rows, "Paper window range is empty."

    return torch.tensor(rows), stride

In [38]:
def all_rows(values, R):
    """Select every row."""

    return torch.ones(len(values), dtype=torch.uint8)

def top_r(values, R):
    """Select the R rows with greatest importance."""

    # Importance ranking
    order = torch.argsort(values, descending=True, stable=True)

    # Top R rows
    mask            = torch.zeros(len(values), dtype=torch.uint8)
    mask[order[:R]] = 1

    return mask

In [39]:
def prepare(layer):
    """Return the selectors of one projection: method -> function(values, R)."""

# 1. READ COST
    n       = layer. in_features
    d       = layer.out_features
    row_kib = d * layer.weight.element_size() / 1024

    # costs[r]: estimated latency of reading r consecutive rows (ms)
    costs = [0.0]

    for r in range(1, n + 1):
        costs.append(latency_table.read_ms(row_kib * r))

    costs = torch.tensor(costs, dtype=torch.float64)

# 2. PAPER
    tuned = tuning.loc[(n, d)]

    sizes, stride = paper_windows(n, row_kib, tuned)
    delays        = costs[sizes]

    def paper(values, R):
        return reader.select_chunks(
            values,
            R,
            sizes,
            delays,
            stride,
            True,  # Sort on the GPU.
        )[0]

# 3. TILES
    B = ceil(tile_reference_kib / row_kib)

    def tiles(values, R):
        return selectors.tiles(
            values,
            R,
            costs,
            B,
        )[0]

# 4. RESULT
    return {
        "Dense"        : all_rows,
        "Top-R"        : top_r,
        "Paper greedy" : paper,
        "Tiles"        : tiles,
    }

In [40]:
# projection name -> its selectors
methods = {}

for name, layer in projections.items():
    methods[name] = prepare(layer)

A condition is a method and a requested fraction of the projection weights.
The paper's sparsity is $1 - f$.

In [41]:
METHODS   = ["Top-R", "Paper greedy", "Tiles"]
FRACTIONS = [0.3, 0.5, 0.7]

# (method, requested fraction)
conditions = [("Dense", 1.0)]

for method in METHODS:
    for fraction in FRACTIONS:
        conditions.append((method, fraction))

## 7. Frame appending

The language model appends the prefix, then one frame per forward pass, to its key–value cache.
As in the paper, rows are selected only in the frame passes; text passes use every row.

In a frame pass, every projection computes the importance of its rows, puts it in the order of its layout,
and asks its selector for the $R$ rows of its budget.
The input channels of all other rows are set to zero.
A zeroed channel adds nothing to the output, so the model computes
what it would compute after reading only the selected rows.

Each projection also keeps the rows it selected last and the time its selector took.
Section 10 reads them.

In [42]:
class Sparsifier:

    """Forward pre-hook of one projection: zero the input channels whose rows are not selected."""

    def __init__(self, select, order, R):
        self.select = select
        self.order  = order
        self.R      = R

    def __call__(self, layer, inputs):

        # Importance of the rows, in their order on flash
        x      = inputs[0]
        values = importance(x)[self.order]

        # Selected rows
        begin = perf_counter_ns()
        rows  = self.select(values, self.R)
        end   = perf_counter_ns()

        # Kept for section 10
        self.rows         = rows
        self.selection_ms = (end - begin) / 1e6

        # Row j on flash belongs to input channel order[j].
        mask             = torch.zeros_like(rows)
        mask[self.order] = rows

        # The returned tuple replaces the inputs of the layer.
        return (x * mask.to(x.device, x.dtype),)

In [43]:
def attach(stack, method, fraction):
    """Hook every projection until the stack closes; return projection name -> hook."""

    layout = LAYOUTS[method]
    hooks  = {}

    for name, layer in projections.items():
        hook = Sparsifier(
            select = methods[name][method],
            order  = orders[layout][name],
            R      = budget_of[fraction][name],
        )
        hooks[name] = hook

        stack.enter_context(layer.register_forward_pre_hook(hook))

    return hooks

In [44]:
def append(cache, tokens):
    """Run the language model on tokens (1, tokens, hidden width), extending the cache."""

    return decoder(
        inputs_embeds   = tokens,
        past_key_values = cache,
        use_cache       = True,
    )

In [45]:
def watch(frame_tokens, method, fraction):
    """Return the cache after the prefix and every frame of one video."""

    cache = DynamicCache()

    # Every row
    append(cache, prefix)

    # Selected rows only
    with ExitStack() as stack:
        attach(stack, method, fraction)

        for tokens in frame_tokens:
            append(cache, tokens)

    return cache

## 8. Replies

After the video, the model reads the rest of the prompt and writes its reply one token at a time.
Each token is the one with the largest logit.

In [46]:
def reply(cache, text, limit=40):
    """Return what the model writes after the video and a text, up to limit tokens."""

    # The cache of the video stays as it is.
    cache  = deepcopy(cache)
    tokens = after_video(text)
    ids    = []

    for _ in range(limit):

        # Most likely next token
        hidden = append(cache, tokens).last_hidden_state
        token  = net.lm_head(hidden[0, -1]).argmax()

        if token == processor.tokenizer.eos_token_id:
            break

        # The model reads its own token next.
        ids.append(int(token))
        tokens = embedding(token[None, None])

    return processor.tokenizer.decode(ids)

A multiple-choice question is followed by an instruction to reply with one option.
The answer we record is the option letter with the largest logit at the first position of the reply.

In [47]:
INSTRUCTION = "\nPlease directly give the best option:"
LETTERS     = "ABCD"
letter_ids  = processor.tokenizer.convert_tokens_to_ids(list(LETTERS))

def answer(cache, question):
    """Return the option letter the model prefers after the video."""

    # Logits at the first position of the reply
    tokens = after_video(question + INSTRUCTION)
    hidden = append(deepcopy(cache), tokens).last_hidden_state
    logits = net.lm_head(hidden[0, -1])

    # One option per line after the first
    options = question.count("\n")
    best    = logits[letter_ids[:options]].argmax()

    return LETTERS[int(best)]

Try both on the example video. This is one of its questions:

In [48]:
example = questions[questions.video_id == example_id].iloc[0]

print(example.question)
print()
print("Right answer:", example.answer)

What is the order of events in the video?
A. A girl is walking on the street while a dog is simultaneously driving
B. A girl is walking on the street, then a dog is driving
C. A dog is driving, then a girl is walking on the street

Right answer: A. A girl is walking on the street while a dog is simultaneously driving


Under every condition, the model watches the video, describes it in up to 40 tokens,
and answers the question.

In [49]:
for method, fraction in conditions:
    cache = watch(example_tokens, method, fraction)

    description = reply(cache, "Describe what happens in this video.")
    letter      = answer(cache, example.question)

    print(f"{method} at {fraction:.0%} answers {letter} and writes:")
    print(f"    {description}")
    print()

Dense at 100% answers A and writes:
    The video begins with a close-up of a dog wearing sunglasses and a hat, sitting in the driver's seat of a car. The background shows a street with cars and a clear sky. The scene

Top-R at 30% answers A and writes:
    The video begins with a humorous and surreal scene where a dog with sunglasses is seen in a car, appearing to be driving while wearing sunglasses. The dog is positioned in the driver's seat, with the

Top-R at 50% answers A and writes:
    The video begins with a humorous and heartwarming scene where a dog with a happy expression, wearing sunglasses, is seen in the driver's seat of a car. The dog's tongue is sticking out

Top-R at 70% answers A and writes:
    The video begins with a close-up of a dog wearing sunglasses and a blue shirt, sitting in the driver's seat of a car. The dog appears to be enjoying the ride, with its tongue out

Paper greedy at 30% answers A and writes:
    The video begins with a humorous and creative seque

## 9. Accuracy

Answer every question under every condition.
The questions of a video share one pass over its frames.
Save one row per question and condition to `answers.csv`.

In [50]:
answer_records = []

for index, (video_id, group) in enumerate(questions.groupby("video_id"), 1):
    frame_tokens = encode(load_video(video_id))

    for method, fraction in conditions:
        cache = watch(frame_tokens, method, fraction)

        for entry in group.itertuples():
            predicted = answer(cache, entry.question)

            answer_records.append({
                # Row of the question in the benchmark table
                "question_id" : entry.Index,

                # Condition
                "method"   : method,
                "fraction" : fraction,

                # Answer
                "predicted" : predicted,
                "correct"   : predicted == entry.answer[0],
            })

    if index % 10 == 0 or index == VIDEOS:
        print(f"{index}/{VIDEOS} videos answered", flush=True)

10/100 videos answered
20/100 videos answered
30/100 videos answered
40/100 videos answered
50/100 videos answered
60/100 videos answered
70/100 videos answered
80/100 videos answered
90/100 videos answered
100/100 videos answered


In [51]:
answers = pd.DataFrame(answer_records)
assert not answers.duplicated(["question_id", "method", "fraction"]).any()

answers.to_csv("answers.csv", index=False)

## 10. Read time

Write the weights of every projection to its own file on flash, one input channel per row,
once in each layout.
Append the frames of the example video again. After each frame pass, read from the files
of the method's layout the rows that the pass selected.

`selection_ms` sums the selector calls of one frame over all projections; `io_ms` sums the reads.
The reader uses six threads and `O_DIRECT`, splitting requests at 768 KiB.
The first two frames warm up the reader; the other 30 are recorded.
Save one row per recorded frame and condition to `frame_times.csv`.

In [52]:
THREADS      = 6    # Reader threads
MAX_READ_KIB = 768  # Largest single read request (KiB)

In [53]:
def write_rows(path, layer, order):
    """Write the weights of one projection to flash, one input channel per row, in an order."""

    rows = layer.weight.T[order.to(device)]
    rows = rows.contiguous().cpu().numpy().tobytes()

    with open(path, "wb") as stream:

        # Reserve the space first. btrfs then stores the bytes uncompressed;
        # it serves O_DIRECT reads of compressed data from the page cache.
        os.posix_fallocate(stream.fileno(), 0, len(rows))

        stream.write(rows)
        stream.flush()
        os.fsync(stream.fileno())

In [54]:
def read_rows(path, layer, mask):
    """Read the selected rows of one projection from flash; return (rows, ms)."""

    row_bytes = layer.out_features * layer.weight.element_size()

    raw, io_us, _, direct = reader.read_rows(
        path,
        mask,
        row_bytes,
        "cpu",                # Where the rows land
        THREADS,
        MAX_READ_KIB * 1024,
        True,                 # Ask for O_DIRECT.
    )

    assert direct, "O_DIRECT unavailable; refusing page-cache timings."

    # Bytes -> (selected rows, d) in the weight dtype
    rows = raw.view(layer.weight.dtype)

    return rows, io_us / 1000

In [55]:
def check(paths, order_of):
    """Check that every file of one layout returns the weights of its projection in that order."""

    for name, layer in projections.items():
        every   = torch.ones(layer.in_features, dtype=torch.uint8)
        rows, _ = read_rows(paths[name], layer, every)

        assert torch.equal(rows, layer.weight.T.cpu()[order_of[name]])

In [56]:
WARMUP = 2  # Leading frames that are not recorded

frame_records = []

In [57]:
def time_frames(frame_tokens, paths, method, fraction):
    """Append the frames of one video and time the reads of each frame."""

    cache = DynamicCache()
    append(cache, prefix)

    with ExitStack() as stack:
        hooks = attach(stack, method, fraction)

        for frame, tokens in enumerate(frame_tokens):
            append(cache, tokens)

            # Rows this frame selected
            selection_ms = 0.0
            io_ms        = 0.0

            for name, hook in hooks.items():
                rows, ms = read_rows(paths[name], projections[name], hook.rows)

                selection_ms += hook.selection_ms
                io_ms        += ms

            if frame >= WARMUP:
                frame_records.append({
                    "method"       : method,
                    "fraction"     : fraction,
                    "frame"        : frame,
                    "selection_ms" : selection_ms,
                    "io_ms"        : io_ms,
                })

In [58]:
with exclusive("experiment05"), TemporaryDirectory(dir=Path.cwd()) as work:

    # layout -> (projection name -> file on flash)
    paths = {}

    for layout, order_of in orders.items():
        paths[layout] = {}

        for index, (name, layer) in enumerate(projections.items()):
            path = str(Path(work) / f"{layout}-{index:03d}.dat")

            write_rows(path, layer, order_of[name])
            paths[layout][name] = path

        check(paths[layout], order_of)

    for method, fraction in conditions:
        time_frames(example_tokens, paths[LAYOUTS[method]], method, fraction)
        print(f"{method} at {fraction:.0%} timed", flush=True)

Dense at 100% timed
Top-R at 30% timed
Top-R at 50% timed
Top-R at 70% timed
Paper greedy at 30% timed
Paper greedy at 50% timed
Paper greedy at 70% timed
Tiles at 30% timed
Tiles at 50% timed
Tiles at 70% timed


In [59]:
frame_times = pd.DataFrame(frame_records)
assert not frame_times.duplicated(["method", "fraction", "frame"]).any()

frame_times.to_csv("frame_times.csv", index=False)